# Closed-Loop Simulation Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/achyutmorang/waymax-simulation-experiments/blob/main/notebooks/closedloop_simulation_colab.ipynb)

## Experiment Report Snapshot

### Research Question
Under equal compute budget, does surprise-guided search discover higher-risk planner failures than risk-only and random search in closed-loop simulation?

### Hypotheses
- `H1`: `joint` and/or `surprise_only` improves risk discovery over `risk_only` and `random`.
- `H2`: surprise signal quality is sufficient for optimization (non-collapsed, low fallback reliance).

### Controlled Factors
- Same per-scenario budget (`budget_evals`) across all methods.
- Common-random-number rollout seeds for fair method comparison.
- Same perturbation feasibility constraints across methods.

### Primary Evaluation Metrics
- `delta_risk` and risk proxy summaries.
- Surprise diagnostics (`surprise_std`, nonzero fraction, fallback ratios).
- Ranking/usefulness metrics (correlation, top-k lift, within-scenario rank consistency).

### Expected Artifacts
- Per-scenario results and per-eval trace CSVs.
- Calibration thresholds/diagnostics artifacts.
- Method summary tables, sanity/fairness checks, and signal analysis outputs.


## Experimental Protocol

1. Bootstrap runtime environment (repo sync, Drive validation, deterministic setup).
2. Configure run identity and sharding.
3. Run diagnostics first (quick probe, preflight, calibration, surprise gate).
4. Run full optimization only when diagnostics pass policy.
5. Export artifacts and run signal-usefulness analyses.

### Update Loop After New Push
1. Re-run bootstrap.
2. Re-run notebook-specific import cell.
3. Re-run from run configuration onward.


## Methodology (Algorithmic Formulation)

For each scenario $s$, we construct a counterfactual perturbation by (i) selecting a high-interaction non-ego target and (ii) applying a behavioral perturbation primitive.

### 1) Interaction-Aware Target Selection

Let $\mathcal{C}_s$ be valid non-ego candidates. We score each candidate $c\in\mathcal{C}_s$ using proximity and interaction risk:

$$
\phi_s(c) = w_d\,\frac{1}{d(c)+\epsilon}
+ w_{\mathrm{ttc}}\,\frac{1}{\min(\mathrm{TTC}(c),H)+\epsilon}
+ w_v\,\mathrm{closing\_speed}(c)
+ w_h\,\mathrm{heading\_conflict}(c).
$$

The perturbation target is

$$
c_s^* = \arg\max_{c\in\mathcal{C}_s} \phi_s(c).
$$

### 2) Behavioral Perturbation Proposal

Given primitive $\pi$ (for example `toward_ego`, `target_brake`, `lateral_left`) and scale $\alpha_k$:

$$
\delta_{s,k}=\Pi_{\mathcal{D}}\!\left(\alpha_k\,g_{\pi}\,u_{\pi}(s,c_s^*) + \eta_k\right),
$$

where $u_{\pi}$ is primitive direction, $g_{\pi}$ is primitive gain, $\eta_k$ is jitter, and projection enforces

$$
\mathcal{D} = \{\delta\in\mathbb{R}^2:\ \|\delta\|_2\le B_2,\ \|\delta\|_\infty\le B_\infty\}.
$$

### 3) Closed-Loop Surprise And Objective

Let $p_t^{\delta}(a)$ and $p_t^{0}(a)$ be proposal/base planner action distributions at step $t$. Surprise is computed over valid paired steps as

$$
S_s(\delta)
= \frac{1}{|\mathcal{T}_{\mathrm{valid}}|}
\sum_{t\in\mathcal{T}_{\mathrm{valid}}}
D\!\left(p_t^{\delta}, p_t^{0}\right),
$$

with divergence channel $D$ selected by `cfg.planner_surprise_name`:

$$
D =
\begin{cases}
W_2\!\left(p_t^{\delta}, p_t^{0}\right), & \texttt{predictive\_w2} \\
\tfrac{1}{2}\left[\mathrm{KL}(p_t^{\delta}\|p_t^{0})+\mathrm{KL}(p_t^{0}\|p_t^{\delta})\right], & \texttt{predictive\_kl}
\end{cases}
$$

If predictive distribution traces are unusable, action-level KL fallback is used.

After calibration normalization, optimization uses

$$
J_{w_R,w_S}(\delta)=w_R\,z_R + w_S\,z_S - \lambda\|\delta\|_2^2,
$$

with equal per-scenario query budget and common-random-number rollout seeds across methods.


## Step 1 - Bootstrap Environment (Repo + Drive + Runtime)

Goal: one idempotent setup cell that prepares runtime infrastructure.

This step automatically:
- syncs latest repo (`main`) into `/content/waymax-simulation-experiments`,
- validates Drive mount + write access,
- runs deterministic setup only if runtime is not already healthy,
- hot-reloads `src.*` module paths.

If setup mutates compiled dependencies, runtime may auto-restart once. After restart, rerun this same cell.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/achyutmorang/waymax-simulation-experiments.git'
REPO_DIR = '/content/waymax-simulation-experiments'
REPO_BRANCH = 'main'
REQUIRED_DRIVE_FOLDER = '/content/drive/MyDrive/waymax_experiments'

runtime_cfg_overrides = {
    'verify_drive_access_every_run': False,
    'force_reinstall': False,
    'auto_restart_after_setup': True,
    'strict_lockfile_check': True,
    'setup_cache_enabled': True,
    'revalidate_core_imports_on_cache_hit': True,
    'setup_cache_path': '/content/.closedloop_setup_cache.json',
    'force_module_hot_reload': True,
}

# Minimal pre-bootstrap so src imports work on first run.
content_root = Path('/content')
content_root.mkdir(parents=True, exist_ok=True)
try:
    _ = os.getcwd()
except FileNotFoundError:
    os.chdir(str(content_root))

if not Path(REPO_DIR).exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-b', REPO_BRANCH, REPO_URL, REPO_DIR], check=True)

for path in (REPO_DIR, f'{REPO_DIR}/src'):
    if path not in sys.path:
        sys.path.insert(0, path)

from src.platform import ColabRuntimeConfig, bootstrap_colab_runtime_with_config

runtime_cfg = ColabRuntimeConfig(
    repo_url=REPO_URL,
    repo_dir=REPO_DIR,
    repo_branch=REPO_BRANCH,
    required_drive_folder=REQUIRED_DRIVE_FOLDER,
    **runtime_cfg_overrides,
)
bootstrap = bootstrap_colab_runtime_with_config(runtime_cfg)

globals()['_CLOSEDLOOP_REPO_REV'] = str(bootstrap.repo_sync.repo_rev)
setup_result = bootstrap.setup.result

print('Working directory:', os.getcwd())
print('Repo commit:', bootstrap.repo_sync.repo_rev)
print('Has src/closedloop:', os.path.exists(os.path.join(REPO_DIR, 'src', 'closedloop')))
print('sys.path[0:3]:', list(bootstrap.repo_sync.sys_path_head[:3]))
if not bootstrap.drive_status.is_colab:
    print('[drive] non-Colab environment; skipping mount')
print('[setup] result:', setup_result)
if bool(setup_result.get('restart_required', False)) and (not bool(runtime_cfg.auto_restart_after_setup)):
    print('[setup] restart_required=True; please restart runtime before running simulation cells.')


### Notebook-Specific API Imports

Keep this cell notebook-specific.
For other experiments, change imports here without touching bootstrap/runtime logic.


In [ ]:
import os
import sys

try:
    from src.workflows import (
        analyze_signal_if_available,
        build_full_simulation_context,
        initialize_run_context,
        report_export_bundle,
        report_main_loop_bundle,
        report_preflight_bundle,
        report_quick_probe_bundle,
        report_run_context,
        report_signal_bundle,
        report_surprise_gate_bundle,
        run_main_loop_with_policy,
        run_preflight_bundle,
        run_quick_probe_with_auto_escalation,
        run_surprise_gate_with_policy,
        summarize_and_export_if_available,
    )
except ModuleNotFoundError as e:
    msg = (
        'Import failed for src.workflows. Re-run bootstrap cell, then rerun this cell.\n'
        f'cwd={os.getcwd()}\n'
        f'sys.path_head={sys.path[:5]}'
    )
    raise RuntimeError(msg) from e


## Step 2 - Configure Run Identity, Persistence, And Auto-Run Policy

This step defines user-level knobs and auto-derives a run plan.

Key controls:
- `RUN_TAG`: explicit run name. Leave empty to auto-adopt recent matching run (if found) or auto-generate timestamped tag.
- `RUN_MODE`: `auto | fresh | resume` (auto infers from existing shard artifacts; with empty `RUN_TAG`, it can adopt recent matching history).
- `PERSIST_ROOT`: persistent Drive root.
- `N_SHARDS`, `SHARD_ID`: workload partition/resume routing.
- auto-run policy for the full closed-loop stage.

`SHARD_ID="auto"` resumes the least-complete shard first.

When resuming, a config drift warning is emitted if current key settings differ from the prior carry-forward config for the same run prefix.


In [ ]:
RUN_MAIN_LOOP_OVERRIDE = None
AUTO_RUN_MAIN_LOOP_WHEN_READY = True

# Leave RUN_TAG empty to auto-adopt most recent matching run under PERSIST_ROOT
# (by RUN_TAG_PREFIX + N_SHARDS). If none exists, auto-generate: <RUN_TAG_PREFIX>_YYYYMMDD_HHMMSS (UTC)
RUN_TAG = ''
RUN_TAG_PREFIX = 'closedloop'
PERSIST_ROOT = '/content/drive/MyDrive/waymax_experiments/closedloop_runs/v1'
N_SHARDS = 5
SHARD_ID = 'auto'
RUN_MODE = 'auto'  # auto | fresh | resume
WARN_ON_CONFIG_DRIFT = True
RESTORE_FROM_UPLOAD = False
SHARD_ID_REQUESTED = SHARD_ID

run_context = initialize_run_context(
    run_tag=RUN_TAG,
    persist_root=PERSIST_ROOT,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
    run_tag_prefix=RUN_TAG_PREFIX,
    auto_generate_run_tag_if_empty=True,
    resume_mode=RUN_MODE,
    warn_on_config_drift=bool(WARN_ON_CONFIG_DRIFT),
)

cfg = run_context.cfg
search_cfg = run_context.search_cfg
ckpt_scan_df = run_context.ckpt_scan_df
shard_progress_df = run_context.shard_progress_df
SHARD_ID = run_context.shard_id
RUN_TAG = run_context.run_tag
run_prefix = run_context.run_prefix

# Interaction-aware behavioral perturbation controls (editable).
cfg.perturb_use_behavioral_proposals = True
cfg.perturb_target_selection_mode = 'highest_interaction'  # highest_interaction | nearest | first_valid
cfg.perturb_target_top_k = 2
cfg.perturb_interaction_ttc_horizon_s = 6.0
cfg.perturb_interaction_w_proximity = 1.0
cfg.perturb_interaction_w_ttc = 1.25
cfg.perturb_interaction_w_closing_speed = 0.35
cfg.perturb_interaction_w_heading_conflict = 0.35
cfg.perturb_behavioral_primitive_cycle = (
    'toward_ego',
    'away_from_ego',
    'target_brake',
    'target_accel',
    'lateral_left',
    'lateral_right',
    'diag_toward_left',
    'diag_toward_right',
)
cfg.perturb_behavioral_longitudinal_gain = 1.05
cfg.perturb_behavioral_lateral_gain = 1.20
cfg.perturb_behavioral_interaction_gain = 1.25
cfg.perturb_behavioral_toward_ego_blend = 0.65

# Predictive surprise divergence channel.
cfg.planner_surprise_name = 'predictive_w2'  # predictive_w2 | predictive_kl | action_kl

if isinstance(SHARD_ID_REQUESTED, str) and SHARD_ID_REQUESTED.strip().lower() == 'auto':
    print(f'[shard] auto-selected SHARD_ID={SHARD_ID}')
else:
    print(f'[shard] using SHARD_ID={SHARD_ID}')

print('[perturb] behavioral proposals enabled =', bool(cfg.perturb_use_behavioral_proposals))
print('[perturb] target selection mode =', cfg.perturb_target_selection_mode)
print('[perturb] primitives =', tuple(cfg.perturb_behavioral_primitive_cycle))
print('[surprise] metric =', cfg.planner_surprise_name)

report_run_context(run_context, display_fn=display)



## Step 3 - Quick Surprise Probe And Tuning

Run a small probe first to test surprise signal viability under low compute.

This step:
- selects high-interaction non-ego targets,
- samples behavioral perturbation primitives with escalation,
- runs probe scenarios/proposals,
- auto-escalates perturbation intensity if collapse is detected,
- applies tuned search settings when a non-collapsed probe is found.



In [ ]:
RUN_QUICK_SURPRISE_PROBE = True
quick_probe_settings = {
    'quick_probe_scenarios': 1,
    'quick_probe_proposals_per_scenario': 4,
    'stop_if_quick_probe_collapsed': False,
    'auto_escalate_quick_probe': True,
    'max_probe_escalations': 3,
    'probe_scale_multipliers': (1.0, 1.35, 1.8),
    'probe_delta_l2_multipliers': (1.0, 1.2, 1.4),
    'probe_delta_clip_multipliers': (1.0, 1.1, 1.2),
    'probe_budget_bump_per_escalation': 2,
    'apply_successful_probe_tuning': True,
}

probe_bundle = run_quick_probe_with_auto_escalation(
    cfg=cfg,
    search_cfg=search_cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
    run_quick_surprise_probe_enabled=bool(RUN_QUICK_SURPRISE_PROBE),
    build_simulation_context=False,
    **quick_probe_settings,
)

cfg = probe_bundle.cfg
search_cfg = probe_bundle.search_cfg
report_quick_probe_bundle(probe_bundle, search_cfg=search_cfg, display_fn=display)


## Step 4 - Build Full WOMD Simulation Context

After probe tuning, build the full iterator and closed-loop run splits for the selected shard.

This step constructs:
- WOMD dataset iterator,
- reference/eval splits,
- scenario handles and base open-loop baseline frame.


In [ ]:
simulation_context = build_full_simulation_context(
    cfg=cfg,
    n_shards=N_SHARDS,
    shard_id=SHARD_ID,
)

runner = simulation_context.runner
eval_idx = simulation_context.eval_idx
reference_df = simulation_context.reference_df
base_eval_openloop_df = simulation_context.base_eval_openloop_df
reference_idx = simulation_context.reference_idx
candidate_eval_idx = simulation_context.candidate_eval_idx

print(f"reference={len(reference_idx)} eval_pool={len(candidate_eval_idx)} eval_shard={len(eval_idx)}")


## Step 5 - Preflight Checks And Closed-Loop Calibration

Preflight validates planner and rollout health before expensive optimization.
Calibration computes thresholds/scales used by objective shaping and gating.

If preflight fails, optimization is skipped until checks pass.


In [ ]:
STOP_ON_PREFLIGHT_FAIL = False

preflight_bundle = run_preflight_bundle(
    runner=runner,
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    reference_df=reference_df,
    restore_from_upload=RESTORE_FROM_UPLOAD,
    stop_on_preflight_fail=bool(STOP_ON_PREFLIGHT_FAIL),
)

READY_FOR_MAIN_LOOP = preflight_bundle.ready_for_main_loop
closedloop_calib_df = preflight_bundle.closedloop_calib_df
closedloop_thresholds = preflight_bundle.closedloop_thresholds
calib_diag_df = preflight_bundle.calib_diag_df
calib_quant_df = preflight_bundle.calib_quant_df
preflight_df = preflight_bundle.preflight_df

report_preflight_bundle(preflight_bundle, display_fn=display)


## Step 6 - Surprise Quality Gate

Gate purpose: block optimization when surprise signal appears degenerate.

Common failure patterns:
- near-zero surprise variance,
- zero nonzero-surprise fraction,
- excessive fallback/proxy behavior.


In [ ]:
SURPRISE_GATE_ENABLED = True
STOP_ON_GATE_FAIL = False
ALLOW_MAIN_LOOP_WHEN_GATE_FAILS = False

gate_bundle = run_surprise_gate_with_policy(
    closedloop_calib_df=closedloop_calib_df,
    ready_for_main_loop=bool(READY_FOR_MAIN_LOOP),
    surprise_gate_enabled=bool(SURPRISE_GATE_ENABLED),
    stop_on_gate_fail=bool(STOP_ON_GATE_FAIL),
    allow_main_loop_when_gate_fails=bool(ALLOW_MAIN_LOOP_WHEN_GATE_FAILS),
)

READY_FOR_OPTIMIZATION = gate_bundle.ready_for_optimization
report_surprise_gate_bundle(gate_bundle, display_fn=display)


## Step 7 - Closed-Loop Optimization Run (Auto-Gated)

Main loop runs only if gate policy allows it.

Override only when intentionally forcing behavior for debugging:
- `RUN_MAIN_LOOP_OVERRIDE=True` forces run,
- `RUN_MAIN_LOOP_OVERRIDE=False` forces skip,
- `None` uses diagnostic policy.


In [ ]:
main_loop_bundle = run_main_loop_with_policy(
    runner=runner,
    eval_idx=eval_idx,
    cfg=cfg,
    search_cfg=search_cfg,
    closedloop_thresholds=closedloop_thresholds,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    ready_for_optimization=bool(READY_FOR_OPTIMIZATION),
    auto_run_main_loop_when_ready=bool(AUTO_RUN_MAIN_LOOP_WHEN_READY),
    run_main_loop_override=RUN_MAIN_LOOP_OVERRIDE,
)

closedloop_results_df = main_loop_bundle.closedloop_results_df
closedloop_trace_df = main_loop_bundle.closedloop_trace_df
report_main_loop_bundle(main_loop_bundle, ready_for_optimization=bool(READY_FOR_OPTIMIZATION))


## Step 8 - Summarize And Export Artifacts

Exports include per-scenario outputs, traces, thresholds, diagnostics, and run metadata.
These are written under `PERSIST_ROOT` and used by evaluation notebooks and paper tables.


In [ ]:
export_bundle = summarize_and_export_if_available(
    cfg=cfg,
    search_cfg=search_cfg,
    eval_idx=eval_idx,
    closedloop_results_df=closedloop_results_df,
    closedloop_trace_df=closedloop_trace_df,
    base_eval_openloop_df=base_eval_openloop_df,
    reference_df=reference_df,
    closedloop_calib_df=closedloop_calib_df,
    preflight_df=preflight_df,
    calib_diag_df=calib_diag_df,
    calib_quant_df=calib_quant_df,
    closedloop_thresholds=closedloop_thresholds,
)

quick_summary_df = export_bundle.quick_summary_df
sanity_df = export_bundle.sanity_df
fairness_checks_df = export_bundle.fairness_checks_df
trace_diag_df = export_bundle.trace_diag_df
artifact_paths = export_bundle.artifact_paths

report_export_bundle(export_bundle, closedloop_results_df=closedloop_results_df, display_fn=display)


## Step 9 - Surprise Signal Usefulness Diagnostics

This step evaluates whether `surprise_pd` is useful for search outcomes (`delta_risk`):
- global correlation,
- quantile-bin trend behavior,
- top-k lift,
- within-scenario ranking consistency.


In [ ]:
signal_bundle = analyze_signal_if_available(
    closedloop_results_df=closedloop_results_df,
    n_bins=10,
    top_fracs=(0.10, 0.20),
    scenario_min_points=3,
)

signal_summary_df = signal_bundle.signal_summary_df
signal_method_corr_df = signal_bundle.signal_method_corr_df
signal_bin_df = signal_bundle.signal_bin_df
signal_topk_lift_df = signal_bundle.signal_topk_lift_df
signal_within_scenario_df = signal_bundle.signal_within_scenario_df

report_signal_bundle(signal_bundle, closedloop_results_df=closedloop_results_df, display_fn=display, preview_rows=20)
